# Imports and Configs

In [ ]:
# Install klib and flash libraries
%pip install klib git+https://github.com/flash-lib/flash.git -q

In [1]:
# Standard Libraries
import toml

# Data Manipulation
import numpy as np
import pandas as pd

# Data Analysis
import seaborn as sns
import matplotlib.pyplot as plt

# Data Preprocessing
import klib
import flash as fz

ModuleNotFoundError: No module named 'toml'

In [ ]:
# Change directory to current working directory
%cd /content/drive/MyDrive/Projects/loan-sanction-prediction

# Initial dataset assessment & preparation

In [ ]:
# Load the dataset
df = pd.read_csv('data/raw/loan_sanction_train.csv')

In [ ]:
# Understand structure of the dataset
df.head()

In [ ]:
# Drop useless features
df.drop('Loan_ID', axis=1, inplace=True)

# Test
print(df.columns)

In [ ]:
# Clean column names
df = klib.clean_column_names(df)

# Test
print(df.columns)

In [ ]:
# Check for duplicate data points
def check_duplicates(df):
    if df.duplicated().any():
        print(df[df.duplicated(keep=False)])
    else:
        print("There are no duplicate data points in the dataframe")

check_duplicates(df)

In [ ]:
# Get some information about the dataset
df.info()

Useful information that we can get from df.info():

- Feature names
- Number of data points
- Number of features
- Data type of features
- Memory usage

In [ ]:
# Extract numerical, categorical, and other features from the dataset
num_cols, cat_cols, other_cols = fz.extract_features(df, 'all', ignore_cols=['loan_status'])

In [ ]:
# Print numerical features of dataset
df[num_cols]

In [ ]:
# Print categorical features of dataset
df[cat_cols]

In [ ]:
# Reorder columns
target_col = ['loan_status']
df = df[num_cols + cat_cols + target_col]

# Test
print(df.columns)

### Conclusions:

- There are no duplicate data points in the dataset
- The dataset contains 614 data points, 11 feature columns, and `loan_status` as the target column.
- Some features are not in appropriate data types. So, adjust the data type accordingly after handling missing values:
    - `applicant_income`: `float`
    - `loan_amount_term` and `credit_history`: `int` then, `object`
    - categorical features: `category` (This will be helpful in analysis)
- There are 3 numerical features:
    - `applicant_income`
    - `coapplicant_income`
    - `loan_amount`
- There are 8 categorical features:
    - `gender`
    - `married`
    - `dependents`
    - `education`
    - `self_employed`
    - `property_area`
    - `loan_amount_term`
    - `credit_history`

### Changes that have made to the dataset in this notebook:

- `Loan_ID` is a useless feature for predictive model building. So, dropped it
- The column names were inconsistent and not in a standard format. So, standardized them using Klib
- Reordered column names to have numerical features at the start and categorical features second

# Initial data analysis

## Outlier analysis

In [ ]:
# Statistical measures
df[num_cols].describe().T

In [ ]:
# Histogram & Box-plot
fig, axs = fz.hist_box_viz(df[num_cols])
fig

#### Conclusions:

- There are many outliers on the upper side of all numerical features, while none are present on the lower side.
- Since the outliers appear to be valid and are not due to data entry issues, we don't have to drop them.
- None of the numerical features follow a normal distribution.

---

#### Handle Outliers:

- Use Tree-Based Models. Because, Tree-Based Models are less sensitive to outliers, so handling outliers may not be necessary if we are using these models.

- Apply feature transformations to make the numerical features more normally distributed, which may reduce the impact of outliers.

- Use IQR-Based Capping to cap outliers to a specific range.

After applying these outlier handling methods, evaluate their impact on the model's performance to determine the most effective approach.

## Missing value analysis

In [ ]:
# Numerical features
num_nan_pct = fz.calc_nan_values(df[num_cols])
num_cols_with_nan = num_nan_pct.index.tolist()

print(num_nan_pct) # Percentage of missing values in numerical features
print(num_cols_with_nan) # Numerical features with missing values

In [ ]:
# Categorical features
cat_nan_pct = fz.calc_nan_values(df[cat_cols])
cat_cols_with_nan = cat_nan_pct.index.tolist()

print(cat_nan_pct) # Percentage of missing values in categorical features
print(cat_cols_with_nan) # Categorical features with missing values

In [ ]:
# Check whether the target column contains any missing values
df['loan_status'].isna().sum()

In [ ]:
# Visualize the distribution of missing values to determine the type of missing values
fig, axs = fz.nan_value_viz(df[num_cols_with_nan + cat_cols_with_nan])
fig

#### Conclusions:

- Only one numerical feature has missing values:
    - `loan_amount`
- Six categorical features have missing values:
    - `gender`
    - `married`
    - `dependents`
    - `self_employed`
    - `loan_amount_term`
    - `credit_history`
- The target column (`loan_status`) doesn't have any missing values.
- Since we have only a few data points, we cannot afford to drop any of them.
- The percentage of missing values is low across all features, so there is no need to drop any columns.
- The missingness of values appears to be random.

---

#### Handle Missing Values:

- Numerical features: Use KNN Imputer or Iterative Imputer.
- Categorical features: Use classifier models to predict missing values.

# Export

In [ ]:
# Export dataset
fz.export(df, 'data/interim/cleaned_train_data_v1.csv', force_overwrite=True)

In [ ]:
# Export metadata as a .toml file to config directory
config_data = {
    "num": {"cols": num_cols,
            "nan": num_cols_with_nan},
    "cat": {"cols": cat_cols,
            "nan": cat_cols_with_nan},
}

with open("config/config.toml", "w") as file:
    toml.dump(config_data, file)

# Test
with open("config/config.toml", "r") as file:
    config_data = toml.load(file)

print(config_data['num']['cols'])

# Data Cleaning Steps:

#### 1. Handle Missing Values:

- Numerical features: Use KNN Imputer or Iterative Imputer.
- Categorical features: Use classifier models to predict missing values.

#### 2. Adjust Data Types:

- `applicant_income`: `float`
- `loan_amount_term` and `credit_history`: `int`, then `str`
- categorical features: `category`

#### 3. Handle Outliers:

- Use Tree-Based Models. Because, Tree-Based Models are less sensitive to outliers, so handling outliers may not be necessary if we are using these models.

- Apply feature transformations to make the numerical features more normally distributed, which may reduce the impact of outliers.

- Use IQR-Based Capping to cap outliers to a specific range.